# Orbit C and C4 retained observations — analysis only
Prepared notebook. Not executed. No model calls. Run privately on Kaggle after setting five attached input environment variables in a private cell. Preserve separate execution identities.

In [ ]:
from pathlib import Path
import json, sys
assert Path('/kaggle/working').is_dir(), 'KAGGLE_ANALYSIS_REQUIRED'
source_files = json.loads(r'''{"orbit_campaign_checkpoint.py":"\"\"\"Analysis-only identity helpers copied from Orbit\u0027s checkpoint module.\n\nThese two pure functions retain the original canonical digest and operational\nfield exclusions. No model, account, resume, generation or transport code exists\nin this module. Source: tools/orbit_campaign_checkpoint.py at Orbit commit\n13b315696bfaf72ac371e5b3498e2ab0d3a9a00e, lines19-30.\n\"\"\"\nimport hashlib\nimport json\n\n\ndef digest(value):\n    encoded = value if isinstance(value, bytes) else json.dumps(value, sort_keys=True, ensure_ascii=False).encode()\n    return hashlib.sha256(encoded).hexdigest()\n\n\ndef campaign_identity(config):\n    operational={\u0027resumeDatasetDir\u0027,\u0027resumeArchives\u0027,\u0027inputCorpusDir\u0027,\u0027inputReferenceDir\u0027,\n                 \u0027workerSecretName\u0027,\u0027quotaSnapshot\u0027,\u0027state\u0027,\u0027quotaChecked\u0027,\u0027modelsChecked\u0027,\n                 \u0027publicRelease\u0027,\u0027modelCallsExecuted\u0027,\u0027softwareTestsExecuted\u0027,\u0027pricingObservation\u0027}\n    return digest({key:value for key,value in config.items() if key not in operational})\n","analyze_kaggle_campaign_results.py":"\"\"\"Aggregate actual Orbit checkpoints inside Kaggle; no model calls.\n\nPublic aggregates contain neither original source text nor reference answers.\nOnly synthetic specification references are scored. Real semantic review stays\npending, and partial conditions are not promoted to complete comparisons.\n\"\"\"\nfrom __future__ import annotations\n\nfrom collections import Counter, defaultdict\nimport itertools\nimport json\nimport math\nfrom pathlib import Path\nimport random\nimport statistics\n\n\ndef _cluster_interval(differences, seed=20261001, repetitions=2000):\n    \"\"\"Paired descriptive interval, resampling whole packets, not questions.\"\"\"\n    if len(differences)<2:\n        return {\u0027clusters\u0027:len(differences),\u0027interval\u0027:None,\u0027reason\u0027:\u0027fewer-than-two-packet-clusters\u0027}\n    rng=random.Random(seed)\n    values=list(differences.values())\n    samples=sorted(statistics.mean(rng.choices(values,k=len(values))) for _ in range(repetitions))\n    return {\u0027clusters\u0027:len(values),\u0027meanDifference\u0027:statistics.mean(values),\n            \u0027interval\u0027:[samples[int(.025*(repetitions-1))],samples[int(.975*(repetitions-1))]],\n            \u0027method\u0027:\u0027paired-packet-cluster-bootstrap-descriptive\u0027,\u0027resamples\u0027:repetitions,\u0027seed\u0027:seed,\n            \u0027limitations\u0027:[\u0027Only six synthetic packets carry reference labels.\u0027,\n                           \u0027No causal or population-wide superiority is certified.\u0027]}\n\n\ndef _attempt_costs(all_rows, identity):\n    \"\"\"Include observed failed/retried generations, without charging a replay twice.\"\"\"\n    from orbit_campaign_checkpoint import digest\n    latest={}\n    for row in all_rows:\n        if row.get(\u0027configurationSha256\u0027)!=identity or row.get(\u0027suite\u0027)!=\u0027C\u0027:\n            continue\n        if row.get(\u0027parameters\u0027,{}).get(\u0027phase\u0027) not in (\u0027extraction\u0027,\u0027production\u0027):\n            continue\n        key=row.get(\u0027key\u0027); attempt=row.get(\u0027attempt\u0027)\n        if not isinstance(key,str) or not isinstance(attempt,int) or isinstance(attempt,bool) or attempt<0:\n            raise RuntimeError(\u0027ANALYSIS_ATTEMPT_IDENTITY_REQUIRED\u0027)\n        identifier=(identity,key,attempt)\n        if row.get(\u0027observedAtUnix\u0027,0)>=latest.get(identifier,{}).get(\u0027observedAtUnix\u0027,0):\n            latest[identifier]=row\n    usages={}; attempts_without_usage=0\n    for row in latest.values():\n        usage=(row.get(\u0027result\u0027) or {}).get(\u0027usage\u0027)\n        if not isinstance(usage,dict) or not usage:\n            attempts_without_usage+=1\n            continue\n        # A restored observation retains its timestamp/content. Distinct\n        # attempts with genuinely new usage remain separately chargeable.\n        observed_at=usage.get(\u0027observedAtUnix\u0027)\n        replay_identity=(observed_at if isinstance(observed_at,(int,float)) and not isinstance(observed_at,bool)\n                         and math.isfinite(observed_at) else (\u0027attempt\u0027,row[\u0027attempt\u0027]))\n        identifier=(identity,row[\u0027key\u0027],replay_identity,digest(usage))\n        usages[identifier]=usage\n    known_components=[]; complete_costs=0; missing_units=2*attempts_without_usage\n    for usage in usages.values():\n        components=[usage.get(name) for name in\n                    (\u0027input_tokens_cost_nanodollars\u0027,\u0027output_tokens_cost_nanodollars\u0027)]\n        valid=[isinstance(value,(int,float)) and not isinstance(value,bool)\n               and math.isfinite(value) and value>=0 for value in components]\n        known_components.extend(value for value,ok in zip(components,valid) if ok)\n        complete_costs+=int(all(valid)); missing_units+=sum(not ok for ok in valid)\n    subtotal=sum(known_components) if known_components else None\n    total=subtotal if usages and missing_units==0 else None\n    return {\u0027generationAttemptRecords\u0027:len(latest),\u0027generations\u0027:len(usages),\n            \u0027generationsWithCost\u0027:complete_costs,\u0027generationsWithMissingCost\u0027:len(usages)-complete_costs,\n            \u0027attemptsWithoutUsage\u0027:attempts_without_usage,\u0027missingCostUnits\u0027:missing_units,\n            \u0027knownSubtotalNanodollars\u0027:subtotal,\u0027totalNanodollars\u0027:total,\n            \u0027knownNanodollars\u0027:subtotal,\u0027missingCostIsNotZero\u0027:True,\n            \u0027costScope\u0027:\u0027observed attempts of the selected C identity; not a projection of unrun tasks\u0027,\n            \u0027attemptStates\u0027:dict(Counter(row.get(\u0027state\u0027,\u0027unspecified\u0027) for row in latest.values())),\n            \u0027deduplication\u0027:\u0027latest identity/key/attempt, then identical usage identity/key/timestamp/content\u0027}\n\n\ndef analyze_c(config, public, gold, ledger_directory):\n    from orbit_campaign_checkpoint import campaign_identity\n    identity=campaign_identity(config)\n    all_rows=[json.loads(p.read_text()) for p in Path(ledger_directory).glob(\u0027*.json\u0027)]\n    rows=[r for r in all_rows if r.get(\u0027configurationSha256\u0027)==identity and r.get(\u0027suite\u0027)==\u0027C\u0027]\n    attempt_rows=[json.loads(p.read_text()) for p in Path(ledger_directory).rglob(\u0027*.json\u0027)]\n    current=[r for r in rows if r.get(\u0027state\u0027)==\u0027completed\u0027]\n    historical=Counter(r.get(\u0027state\u0027,\u0027unspecified\u0027) for r in all_rows if r.get(\u0027configurationSha256\u0027)!=identity)\n    phases=Counter(r.get(\u0027parameters\u0027,{}).get(\u0027phase\u0027) for r in current)\n    questions=defaultdict(list)\n    for q in public[\u0027questions\u0027]:questions[q[\u0027packet\u0027]].append(q[\u0027id\u0027])\n    targets={identifier:value.get(\u0027modelDecision\u0027,value[\u0027decision\u0027]) for identifier,value in gold.items()}\n    interpreted={tuple(r[\u0027parameters\u0027][k] for k in (\u0027model\u0027,\u0027packet\u0027,\u0027repetition\u0027,\u0027condition\u0027)):r[\u0027result\u0027]\n                 for r in current if r.get(\u0027parameters\u0027,{}).get(\u0027phase\u0027)==\u0027production-interpretation\u0027}\n    productions={tuple(r[\u0027parameters\u0027][k] for k in (\u0027model\u0027,\u0027packet\u0027,\u0027repetition\u0027,\u0027condition\u0027)):r\n                 for r in rows if r.get(\u0027parameters\u0027,{}).get(\u0027phase\u0027)==\u0027production\u0027}\n    metrics=[]; paired=defaultdict(dict)\n    for model in config[\u0027models\u0027]:\n        for condition in (\u0027none\u0027,\u0027baseline\u0027,\u0027n\u0027,\u0027p\u0027):\n            counts=Counter();confusion=Counter();packet_counts=defaultdict(Counter);missing_states=Counter()\n            for packet in range(1,11):\n                for repetition in range(3):\n                    output=interpreted.get((model,packet,repetition,condition))\n                    if output is None:\n                        counts[\u0027missingProductions\u0027]+=1\n                        production=productions.get((model,packet,repetition,condition))\n                        state=production.get(\u0027state\u0027,\u0027unspecified\u0027) if production else \u0027never-run\u0027\n                        missing_states[state]+=1\n                        category=(\u0027neverRunProductions\u0027 if production is None else\n                                  \u0027inProgressProductions\u0027 if state in (\u0027running\u0027,\u0027observing\u0027) else\n                                  \u0027uninterpretedProductions\u0027 if state==\u0027completed\u0027 else \u0027failedProductions\u0027)\n                        counts[category]+=1\n                        continue\n                    counts[\u0027observedProductions\u0027]+=1\n                    answers=output.get(\u0027answer\u0027,{}).get(\u0027results\u0027,[]) if output.get(\u0027status\u0027)==\u0027completed\u0027 else []\n                    lookup={a.get(\u0027questionId\u0027):a for a in answers if isinstance(a,dict)}\n                    if output.get(\u0027status\u0027)!=\u0027completed\u0027:counts[\u0027invalidProductions\u0027]+=1\n                    for identifier in questions[packet]:\n                        if identifier not in targets:\n                            counts[\u0027realDecisionsUnscored\u0027]+=1\n                            continue\n                        reference=targets[identifier]\n                        decision=lookup.get(identifier,{}).get(\u0027decision\u0027,\u0027INVALID\u0027)\n                        if decision not in (\u0027ADMIT\u0027,\u0027REJECT\u0027,\u0027HOLD\u0027):decision=\u0027INVALID\u0027\n                        counts[\u0027syntheticDecisionsObserved\u0027]+=1\n                        correct=decision==reference\n                        counts[\u0027syntheticCorrect\u0027]+=int(correct)\n                        counts[\u0027invalidSyntheticDecisions\u0027]+=int(decision==\u0027INVALID\u0027)\n                        counts[\u0027necessaryHold\u0027]+=int(reference==\u0027HOLD\u0027)\n                        counts[\u0027necessaryHoldUsed\u0027]+=int(reference==\u0027HOLD\u0027 and decision==\u0027HOLD\u0027)\n                        counts[\u0027excessiveHold\u0027]+=int(reference!=\u0027HOLD\u0027 and decision==\u0027HOLD\u0027)\n                        counts[\u0027unjustifiedConclusion\u0027]+=int(reference==\u0027HOLD\u0027 and decision in (\u0027ADMIT\u0027,\u0027REJECT\u0027))\n                        confusion[(reference,decision)]+=1\n                        packet_counts[packet][\u0027correct\u0027]+=int(correct)\n                        packet_counts[packet][\u0027total\u0027]+=1\n                        paired[(model,packet,repetition,identifier)][condition]=int(correct)\n            observed=counts[\u0027syntheticDecisionsObserved\u0027]\n            metrics.append({\u0027model\u0027:model,\u0027condition\u0027:condition,\u0027counts\u0027:dict(counts),\n                \u0027missingProductionStates\u0027:dict(missing_states),\n                \u0027syntheticAccuracyObserved\u0027:counts[\u0027syntheticCorrect\u0027]/observed if observed else None,\n                \u0027plannedProductions\u0027:30,\u0027plannedSyntheticDecisions\u0027:108,\u0027plannedRealDecisionsUnscored\u0027:72,\n                \u0027complete\u0027:counts[\u0027observedProductions\u0027]==30,\n                \u0027confusion\u0027:[{\u0027referenceCategory\u0027:a,\u0027predictedCategory\u0027:b,\u0027count\u0027:n} for (a,b),n in sorted(confusion.items())],\n                \u0027packets\u0027:[{\u0027packet\u0027:p,**dict(v)} for p,v in sorted(packet_counts.items())]})\n    comparisons=[]\n    for model in config[\u0027models\u0027]:\n        for a,b in itertools.combinations((\u0027none\u0027,\u0027baseline\u0027,\u0027n\u0027,\u0027p\u0027),2):\n            differences=defaultdict(list)\n            for (observed_model,packet,repetition,question),values in paired.items():\n                if observed_model==model and a in values and b in values:\n                    differences[packet].append(values[b]-values[a])\n            means={packet:statistics.mean(values) for packet,values in differences.items()}\n            comparisons.append({\u0027model\u0027:model,\u0027from\u0027:a,\u0027to\u0027:b,\n                \u0027matchedSyntheticDecisions\u0027:sum(map(len,differences.values())),\n                \u0027plannedMatchedSyntheticDecisions\u0027:108,\u0027complete\u0027:sum(map(len,differences.values()))==108,\n                **_cluster_interval(means)})\n    return {\u0027format\u0027:\u0027orbit-c-actual-checkpoint-analysis-v1\u0027,\u0027host\u0027:\u0027Kaggle\u0027,\u0027modelCallsDuringAnalysis\u0027:0,\n        \u0027campaignId\u0027:config[\u0027campaignId\u0027],\u0027configurationSha256\u0027:identity,\n        \u0027execution\u0027:{\u0027plannedExtractions\u0027:60,\u0027observedExtractions\u0027:phases[\u0027extraction\u0027],\n                     \u0027plannedProductions\u0027:240,\u0027observedProductions\u0027:phases[\u0027production\u0027],\n                     \u0027completedPackets\u0027:phases[\u0027packet\u0027],\n                     \u0027currentStates\u0027:dict(Counter(r.get(\u0027state\u0027,\u0027unspecified\u0027) for r in rows)),\n                     \u0027complete\u0027:phases[\u0027extraction\u0027]==60 and phases[\u0027production\u0027]==240 and phases[\u0027packet\u0027]==60},\n        \u0027historicalStatesNotPooled\u0027:dict(historical),\u0027metrics\u0027:metrics,\u0027pairedComparisons\u0027:comparisons,\n        \u0027observedCosts\u0027:_attempt_costs(attempt_rows,identity),\n        \u0027referenceStatus\u0027:{\u0027syntheticQuestions\u0027:36,\u0027realQuestionsPendingHuman\u0027:24},\n        \u0027limitations\u0027:[\u0027Synthetic specification accuracy does not establish real-source semantic truth.\u0027,\n                      \u0027Exact quotes and citations require a separate relevance audit.\u0027,\n                      \u0027Partial runs remain partial; completed-condition selection is not a final ranking.\u0027,\n                      \u0027No engine is selected or activated automatically.\u0027]}\n\n\ndef write_c_analysis(config,public,gold,ledger_directory,output):\n    if not Path(\u0027/kaggle/working\u0027).is_dir():raise RuntimeError(\u0027KAGGLE_ANALYSIS_REQUIRED\u0027)\n    result=analyze_c(config,public,gold,ledger_directory)\n    Path(output).write_text(json.dumps(result,indent=2)+\u0027\\n\u0027,encoding=\u0027utf-8\u0027)\n    print(json.dumps({\u0027execution\u0027:result[\u0027execution\u0027],\u0027referenceStatus\u0027:result[\u0027referenceStatus\u0027],\n                      \u0027observedCosts\u0027:result[\u0027observedCosts\u0027],\u0027modelCallsDuringAnalysis\u0027:0}))\n    return result\n","run_analysis_only.py":"\"\"\"Kaggle-only analysis of two retained identities; no model or network calls.\n\nSupply private attached input paths at runtime. This source contains no private\ndataset identifiers, answers, snapshots or operational credentials.\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport tempfile\nimport zipfile\n\nimport analyze_kaggle_campaign_results as analysis\nfrom orbit_campaign_checkpoint import campaign_identity, digest\n\nC_ID = \"0d433497a1e6fa7810c68a51709e821d204f3241707e39850e45b2ce6283ad85\"\nC4_ID = \"feb51fee0843163946fa85f39e1d450db5960a0375c9c989f60a55c86e433177\"\nC_ARCHIVE_SHA = \"55276f8c2d772d1c87caec867e0678a7d16d9eec2b97e09beea1ba471c0f3116\"\nC4_ARCHIVE_SHA = \"3d5ae48fb3900fe04eab8b2a625fee49301adf4d787a9f3079c2ec485ee8326b\"\nCORPUS_SHA = \"c8d7f865ba2c9667f43759a053174e3250150704746141e52fc0dca236190f5a\"\nGOLD_SHA = \"a0fffdf104aa36b3e994c57902ffc7a76f2d1a1fd143cc821ca60a4c88e93b23\"\nANALYZER_SHA = \"1ae7a80aaf7fcfbec2e1121af6a73f30777b26976400308bedb7d5e30ecc21d3\"\nEXTRACTION_KEY = \"3c3161accc4df1068312b3d1395c5017bdee9fe3a4c7223ceeb290c855113196\"\nEXTRACTION_RAW_SHA = \"6fc132ef4784e4345e09d87f5b94fc7989f3d871d4b9a986211d8807ab6731bb\"\n\n\ndef private_input(variable):\n    path = Path(os.environ[variable]).resolve()\n    if not path.is_relative_to(Path(\"/kaggle/input\")) or not path.is_file():\n        raise RuntimeError(\"PRIVATE_ATTACHED_INPUT_REQUIRED\")\n    return path\n\n\ndef verified_json(path, expected):\n    data = path.read_bytes()\n    if hashlib.sha256(data).hexdigest() != expected:\n        raise RuntimeError(\"INPUT_FINGERPRINT_MISMATCH\")\n    return json.loads(data)\n\n\ndef open_archive(path, expected):\n    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:\n        raise RuntimeError(\"ARCHIVE_FINGERPRINT_MISMATCH\")\n    archive = zipfile.ZipFile(path)\n    names = archive.namelist()\n    if len(names) != len(set(names)):\n        raise RuntimeError(\"DUPLICATE_ARCHIVE_MEMBER\")\n    if sum(item.file_size for item in archive.infolist()) > 100_000_000:\n        raise RuntimeError(\"ARCHIVE_EXPANSION_BOUND_EXCEEDED\")\n    for name in names:\n        parts = PurePosixPath(name).parts\n        if name.startswith(\"/\") or \"\\\\\" in name or \"..\" in parts or \":\" in name:\n            raise RuntimeError(\"UNSAFE_ARCHIVE_MEMBER\")\n    return archive\n\n\ndef projected_ledger(archive, prefix, identity, directory):\n    \"\"\"Retain only numeric usage and categorical answers in temporary files.\"\"\"\n    rows = []\n    for name in archive.namelist():\n        relative = name.removeprefix(prefix + \"/\")\n        if not name.startswith(prefix + \"/\") or not relative.endswith(\".json\"):\n            continue\n        row = json.loads(archive.read(name))\n        if row.get(\"suite\") != \"C\" or row.get(\"configurationSha256\") != identity:\n            continue\n        result = row.get(\"result\") or {}\n        projection = {key: row.get(key) for key in (\n            \"key\", \"configurationSha256\", \"campaignId\", \"suite\", \"parameters\",\n            \"attempt\", \"state\", \"observedAtUnix\")}\n        usage = result.get(\"usage\")\n        safe_usage = ({key: value for key, value in usage.items()\n                       if key in (\"observedAtUnix\", \"input_tokens\", \"output_tokens\",\n                                  \"input_tokens_cost_nanodollars\", \"output_tokens_cost_nanodollars\",\n                                  \"total_backend_latency_ms\")}\n                      if isinstance(usage, dict) else None)\n        safe_result = {\"status\": result.get(\"status\"), \"usage\": safe_usage}\n        safe_result[\"answer\"] = {\"results\": [\n            {\"questionId\": value.get(\"questionId\"), \"decision\": value.get(\"decision\")}\n            for value in (result.get(\"answer\") or {}).get(\"results\", [])\n            if isinstance(value, dict)\n        ]}\n        projection[\"result\"] = safe_result\n        rows.append(projection)\n    current = {}\n    for row in rows:\n        key = row[\"key\"]\n        if not isinstance(key, str) or not isinstance(row[\"attempt\"], int):\n            raise RuntimeError(\"CHECKPOINT_IDENTITY_REQUIRED\")\n        rank = (row[\"attempt\"], row.get(\"observedAtUnix\") or 0)\n        if key not in current or rank > current[key][0]:\n            current[key] = (rank, row)\n    (directory / \"attempts\").mkdir(parents=True)\n    for index, row in enumerate(rows):\n        (directory / \"attempts\" / f\"{index}.json\").write_text(json.dumps(row), encoding=\"utf-8\")\n    for key, (_, row) in current.items():\n        (directory / f\"{key}.json\").write_text(json.dumps(row), encoding=\"utf-8\")\n    return [row for _, row in current.values()]\n\n\ndef run():\n    if not Path(\"/kaggle/working\").is_dir():\n        raise RuntimeError(\"KAGGLE_ANALYSIS_REQUIRED\")\n    if hashlib.sha256(Path(analysis.__file__).read_bytes()).hexdigest() != ANALYZER_SHA:\n        raise RuntimeError(\"ANALYZER_SOURCE_MISMATCH\")\n    config = json.loads(private_input(\"ORBIT_C_CONFIG\").read_bytes())\n    if campaign_identity(config) != C_ID:\n        raise RuntimeError(\"C_CONFIGURATION_MISMATCH\")\n    public = verified_json(private_input(\"ORBIT_PUBLIC_INPUT\"), CORPUS_SHA)\n    gold = verified_json(private_input(\"ORBIT_PRIVATE_GOLD\"), GOLD_SHA)\n    if len(public[\"questions\"]) != 60 or len(gold) != 36:\n        raise RuntimeError(\"REFERENCE_SCOPE_MISMATCH\")\n    output = Path(\"/kaggle/working/orbit-analysis-public\")\n    output.mkdir(exist_ok=True)\n    with open_archive(private_input(\"ORBIT_C_ARCHIVE\"), C_ARCHIVE_SHA) as parent_archive, \\\n         open_archive(private_input(\"ORBIT_C4_ARCHIVE\"), C4_ARCHIVE_SHA) as complement_archive, \\\n         tempfile.TemporaryDirectory(prefix=\"orbit-analysis-\", dir=\"/kaggle/temp\") as temporary:\n        extraction = json.loads(parent_archive.read(f\"orbit-campaign-v2/C/{EXTRACTION_KEY}.json\"))\n        if (extraction.get(\"configurationSha256\") != C_ID or extraction.get(\"state\") != \"completed\"\n                or hashlib.sha256(extraction[\"result\"][\"raw\"].encode()).hexdigest() != EXTRACTION_RAW_SHA):\n            raise RuntimeError(\"SHARED_EXTRACTION_MISMATCH\")\n        c4_config = json.loads(complement_archive.read(\"complement-manifest.json\"))\n        if campaign_identity(c4_config) != C4_ID:\n            raise RuntimeError(\"C4_CONFIGURATION_MISMATCH\")\n        if c4_config[\"complement\"][\"parentConfigurationSha256\"] != C_ID:\n            raise RuntimeError(\"C4_PARENT_MISMATCH\")\n        c_directory, c4_directory = Path(temporary) / \"C\", Path(temporary) / \"C4\"\n        c_directory.mkdir(); c4_directory.mkdir()\n        projected_ledger(parent_archive, \"orbit-campaign-v2/C\", C_ID, c_directory)\n        c4_rows = projected_ledger(complement_archive, \"C\", C4_ID, c4_directory)\n        c_result = analysis.analyze_c(config, public, gold, c_directory)\n        if (c_result[\"execution\"][\"observedExtractions\"] != 60\n                or c_result[\"execution\"][\"observedProductions\"] != 236\n                or c_result[\"execution\"][\"completedPackets\"] != 59):\n            raise RuntimeError(\"C_RECEIPT_COVERAGE_MISMATCH\")\n        target = c4_config[\"complement\"][\"target\"]\n        for row in c4_rows:\n            if any(row[\"parameters\"].get(key) != value for key, value in target.items()):\n                raise RuntimeError(\"C4_OUTSIDE_DECLARED_TARGET\")\n        c4_base = analysis.analyze_c(c4_config, public, gold, c4_directory)\n        if (c4_base[\"execution\"][\"observedProductions\"] != 4\n                or c4_base[\"execution\"][\"completedPackets\"] != 1):\n            raise RuntimeError(\"C4_RECEIPT_COVERAGE_MISMATCH\")\n        if len([question for question in public[\"questions\"]\n                if question[\"packet\"] == target[\"packet\"] and question[\"id\"] in gold]) != 6:\n            raise RuntimeError(\"C4_REFERENCE_SCOPE_MISMATCH\")\n        c4_metrics = []\n        for metric in c4_base[\"metrics\"]:\n            if metric[\"model\"] != target[\"model\"]:\n                continue\n            scoped_counts = {key: value for key, value in metric[\"counts\"].items()\n                             if key not in (\"missingProductions\", \"neverRunProductions\",\n                                            \"failedProductions\", \"inProgressProductions\",\n                                            \"uninterpretedProductions\")}\n            c4_metrics.append({\"model\": metric[\"model\"], \"condition\": metric[\"condition\"],\n                               \"counts\": scoped_counts, \"confusion\": metric[\"confusion\"],\n                               \"syntheticAccuracyObserved\": metric[\"syntheticAccuracyObserved\"],\n                               \"plannedProductionsWithinComplement\": 1,\n                               \"plannedSyntheticDecisionsWithinComplement\": 6,\n                               \"completeWithinComplement\": scoped_counts.get(\"observedProductions\") == 1})\n        c4_result = {\"format\": \"orbit-c4-analysis-v1\", \"host\": \"Kaggle\",\n                     \"modelCallsDuringAnalysis\": 0, \"configurationSha256\": C4_ID,\n                     \"parentConfigurationSha256\": C_ID, \"target\": target,\n                     \"execution\": {\"newExtractions\": 0, \"productions\": 4, \"completedPackets\": 1},\n                     \"metrics\": c4_metrics, \"observedCosts\": c4_base[\"observedCosts\"],\n                     \"limitations\": [\"One synthetic packet; no standalone population-level comparison.\",\n                                     \"Separate complement identity; not pooled into historical C.\"]}\n        c_result[\"sourceArchiveSha256\"] = C_ARCHIVE_SHA\n        c4_result[\"sourceArchiveSha256\"] = C4_ARCHIVE_SHA\n        (output / \"c-resumption-analysis.json\").write_text(json.dumps(c_result, indent=2) + \"\\n\")\n        (output / \"c4-complement-analysis.json\").write_text(json.dumps(c4_result, indent=2) + \"\\n\")\n        summary = {\"host\": \"Kaggle\", \"modelCallsDuringAnalysis\": 0, \"pooledAsOneExecution\": False,\n                   \"C\": c_result[\"execution\"], \"C4\": c4_result[\"execution\"],\n                   \"realQuestionsPendingHuman\": 24,\n                   \"accuracyByIdentity\": {\"C\": [\n                       {key: metric[key] for key in (\"model\", \"condition\", \"syntheticAccuracyObserved\", \"counts\")}\n                       for metric in c_result[\"metrics\"]], \"C4\": c4_metrics}}\n        (output / \"analysis-summary.json\").write_text(json.dumps(summary, indent=2) + \"\\n\")\n        print(json.dumps(summary, indent=2))\n    return summary\n\n\nif __name__ == \"__main__\":\n    run()\n"}''')
source_directory = Path('/kaggle/temp/orbit-analysis-source')
source_directory.mkdir(parents=True, exist_ok=True)
for name, source in source_files.items():
    (source_directory / name).write_text(source, encoding='utf-8')
sys.path.insert(0, str(source_directory))


In [ ]:
from run_analysis_only import run
summary = run()
